In [ ]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'parameters'))
sys.path.insert(0, os.getcwd())

from analysis_utils import (
    load_best_runs,
    load_top_n_runs,
    extract_history,
    add_wall_times,
    build_summary_dataframe,
    plot_convergence_curves,
    plot_multi_function_performance,
    print_convergence_summary,
    print_best_hyperparameters,
    save_best_hyperparameters,
    get_optimizer_colors,
    ALL_OPTIMIZERS,
)

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuration


In [ ]:
# Backend: "wandb" or "local"
BACKEND = "local"

# WandB settings (ignored if BACKEND == "local")
PROJECT = "induced_metric"
ENTITY = "thomas_harvey"

# Results directory (for local backend)
RESULTS_DIR = os.path.join('..', 'results')

# Test functions to analyse
FUNCTIONS = ['beale', 'rosenbrock', 'himmelblau', 'ackley', 'rastrigin']

# Auto-detect which optimizers have results for at least one function
if BACKEND == "local":
    _available = set()
    for func_name in FUNCTIONS:
        task_dir = os.path.join(RESULTS_DIR, f"small_examples_{func_name}")
        if os.path.isdir(task_dir):
            _available.update(os.listdir(task_dir))
    # Keep the canonical ordering from ALL_OPTIMIZERS
    OPTIMIZERS = [o for o in ALL_OPTIMIZERS if o in _available]
else:
    OPTIMIZERS = ALL_OPTIMIZERS

# Iteration/batch number to load results from
ITERATION = 1

colors = get_optimizer_colors(OPTIMIZERS)
print(f"Backend: {BACKEND}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Functions: {FUNCTIONS}")
print(f"Iteration: {ITERATION}")
print(f"Optimizers ({len(OPTIMIZERS)}): {OPTIMIZERS}")

# Load Results


In [ ]:
# Load best runs for each function
all_best_runs = {}
for func_name in FUNCTIONS:
    task_tag = f"small_examples_{func_name}"
    print(f"\n--- {func_name.upper()} ---")

    best_runs = load_best_runs(
        backend=BACKEND,
        optimizers=OPTIMIZERS,
        task_tag=task_tag,
        project=PROJECT,
        entity=ENTITY,
        results_dir=RESULTS_DIR,
        metric_key="sweep_metric",
        direction="minimize",
        sort_metric="sweep_metric",
        sort_order="+",
        iteration=ITERATION,
    )
    all_best_runs[func_name] = best_runs

print(f"\nLoaded results for {len(all_best_runs)} functions")
for func_name, runs in all_best_runs.items():
    print(f"  {func_name}: {len(runs)} optimizers")

# Performance Comparison


In [ ]:
summary_df = build_summary_dataframe(all_best_runs, optimizers=OPTIMIZERS)

if len(summary_df) == 0:
    print("No results found. Check RESULTS_DIR and ITERATION settings.")
else:
    print(f"Total results: {len(summary_df)}")
    print(f"Convergence rate: {summary_df['converged'].mean():.1%}")

    print("\nConvergence by optimizer:")
    conv = summary_df.groupby('optimizer')['converged'].agg(['count', 'sum', 'mean'])
    conv.columns = ['runs', 'converged', 'rate']
    print(conv.to_string())

In [ ]:
if len(summary_df) == 0:
    print("No results to plot.")
else:
    fig = plot_multi_function_performance(summary_df)
    plt.show()

# Convergence Curves


In [ ]:
for func_name in FUNCTIONS:
    best_runs = all_best_runs[func_name]
    if not best_runs:
        continue

    history_data = extract_history(
        BACKEND, best_runs, ["function_value", "runtime_seconds"]
    )
    add_wall_times(history_data, time_key="runtime_seconds")

    fig = plot_convergence_curves(
        history_data,
        y_keys=["function_value"],
        x_keys=["epoch", "wall_times"],
        best_runs=best_runs,
        converged_key="converged",
        best_epoch_key="iterations",
        colors=colors,
        title=f'Convergence: {func_name.title()} Function',
        y_labels={"function_value": "Function Value"},
        x_labels={"epoch": "Iterations", "wall_times": "Runtime (s)"},
    )
    plt.show()

    print_convergence_summary(best_runs, func_name=func_name, optimizers=OPTIMIZERS)

# Rankings


In [ ]:
if len(summary_df) == 0:
    print("No results to rank.")
else:
    print("=" * 80)
    print("OVERALL RANKINGS")
    print("=" * 80)

    print("\n1. BY CONVERGENCE RATE:")
    conv_rate = summary_df.groupby('optimizer')['converged'].mean().sort_values(ascending=False)
    for i, (opt, rate) in enumerate(conv_rate.items(), 1):
        print(f"  {i:2d}. {opt:25s}: {rate:.1%}")

    converged_only = summary_df[summary_df['converged'] == True]

    if len(converged_only) > 0:
        print("\n2. BY AVERAGE ITERATIONS (converged runs):")
        avg_iter = converged_only.groupby('optimizer')['iterations'].mean().sort_values()
        for i, (opt, val) in enumerate(avg_iter.items(), 1):
            print(f"  {i:2d}. {opt:25s}: {val:.1f}")

        print("\n3. BY AVERAGE RUNTIME (converged runs):")
        avg_rt = converged_only.groupby('optimizer')['runtime_ms'].mean().sort_values()
        for i, (opt, val) in enumerate(avg_rt.items(), 1):
            print(f"  {i:2d}. {opt:25s}: {val:.1f} ms")
    else:
        print("\nNo converged runs found.")

# Best Hyperparameters


In [ ]:
# Print and save best hyperparameters per function
all_hp_rows = []

for func_name in FUNCTIONS:
    best_runs = all_best_runs[func_name]
    if not best_runs:
        continue

    print(f"\n{'=' * 60}")
    print(f"{func_name.upper()} FUNCTION")
    print(f"{'=' * 60}")
    print_best_hyperparameters(best_runs)

    for opt, run_info in best_runs.items():
        row = {
            'function': func_name,
            'optimizer': opt,
            'run_name': run_info.get('name', 'unknown'),
        }
        row.update(run_info.get('config', {}))
        row.update(run_info.get('summary', {}))
        all_hp_rows.append(row)

df = pd.DataFrame(all_hp_rows)
filename = f'best_hyperparameters_SmallExamples_itr_{ITERATION}.csv'
df.to_csv(filename, index=False)
print(f"\nSaved all hyperparameters to {filename}")
df